<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Investigacion_Avanzada/Fluidos_Shallow_Water.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Fluidos Shallow Water

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def shallow_water_solver():
    # Domain and grid
    Nx, Ny = 200, 200
    Lx, Ly = 10.0, 10.0
    dx, dy = Lx/Nx, Ly/Ny
    x = np.linspace(-Lx/2, Lx/2, Nx)
    y = np.linspace(-Ly/2, Ly/2, Ny)
    X, Y = np.meshgrid(x, y)
    
    g = 9.81
    dt = 0.001
    steps = 800
    
    # Initial state
    h = np.ones((Ny, Nx)) # Water depth
    u = np.zeros((Ny, Nx)) # x-velocity
    v = np.zeros((Ny, Nx)) # y-velocity
    
    # Droplet perturbation
    R = np.sqrt(X**2 + Y**2)
    h += 0.5 * np.exp(-R**2 / 0.5)
    
    # Arrays for MacCormack scheme
    h_new = np.zeros_like(h)
    u_new = np.zeros_like(u)
    v_new = np.zeros_like(v)
    
    for n in range(steps):
        # Fluxes
        F1 = h * u
        F2 = h * u**2 + 0.5 * g * h**2
        F3 = h * u * v
        
        G1 = h * v
        G2 = h * u * v
        G3 = h * v**2 + 0.5 * g * h**2
        
        # Predictor step (forward differences)
        hp = h.copy()
        up = u.copy()
        vp = v.copy()
        
        # Internal points predictor
        hp[1:-1, 1:-1] = h[1:-1, 1:-1] - dt/dx * (F1[1:-1, 2:] - F1[1:-1, 1:-1]) - dt/dy * (G1[2:, 1:-1] - G1[1:-1, 1:-1])
        up[1:-1, 1:-1] = (h[1:-1, 1:-1]*u[1:-1, 1:-1] - dt/dx * (F2[1:-1, 2:] - F2[1:-1, 1:-1]) - dt/dy * (G2[2:, 1:-1] - G2[1:-1, 1:-1])) / hp[1:-1, 1:-1]
        vp[1:-1, 1:-1] = (h[1:-1, 1:-1]*v[1:-1, 1:-1] - dt/dx * (F3[1:-1, 2:] - F3[1:-1, 1:-1]) - dt/dy * (G3[2:, 1:-1] - G3[1:-1, 1:-1])) / hp[1:-1, 1:-1]
        
        # Boundaries (reflective)
        hp[0,:] = hp[1,:]; hp[-1,:] = hp[-2,:]; hp[:,0] = hp[:,1]; hp[:,-1] = hp[:,-2]
        up[0,:] = -up[1,:]; up[-1,:] = -up[-2,:]; up[:,0] = -up[:,1]; up[:,-1] = -up[:,-2]
        vp[0,:] = -vp[1,:]; vp[-1,:] = -vp[-2,:]; vp[:,0] = -vp[:,1]; vp[:,-1] = -vp[:,-2]
        
        # Predictor fluxes
        F1p = hp * up
        F2p = hp * up**2 + 0.5 * g * hp**2
        F3p = hp * up * vp
        
        G1p = hp * vp
        G2p = hp * up * vp
        G3p = hp * vp**2 + 0.5 * g * hp**2
        
        # Corrector step (backward differences)
        hc = 0.5 * (h + hp)
        uc = np.zeros_like(u)
        vc = np.zeros_like(v)
        
        h_new[1:-1, 1:-1] = 0.5 * (h[1:-1, 1:-1] + hp[1:-1, 1:-1] - dt/dx * (F1p[1:-1, 1:-1] - F1p[1:-1, :-2]) - dt/dy * (G1p[1:-1, 1:-1] - G1p[:-2, 1:-1]))
        
        hu_new = 0.5 * (h[1:-1, 1:-1]*u[1:-1, 1:-1] + hp[1:-1, 1:-1]*up[1:-1, 1:-1] - dt/dx * (F2p[1:-1, 1:-1] - F2p[1:-1, :-2]) - dt/dy * (G2p[1:-1, 1:-1] - G2p[:-2, 1:-1]))
        hv_new = 0.5 * (h[1:-1, 1:-1]*v[1:-1, 1:-1] + hp[1:-1, 1:-1]*vp[1:-1, 1:-1] - dt/dx * (F3p[1:-1, 1:-1] - F3p[1:-1, :-2]) - dt/dy * (G3p[1:-1, 1:-1] - G3p[:-2, 1:-1]))
        
        u_new[1:-1, 1:-1] = hu_new / h_new[1:-1, 1:-1]
        v_new[1:-1, 1:-1] = hv_new / h_new[1:-1, 1:-1]
        
        # Apply boundaries
        h_new[0,:] = h_new[1,:]; h_new[-1,:] = h_new[-2,:]; h_new[:,0] = h_new[:,1]; h_new[:,-1] = h_new[:,-2]
        u_new[0,:] = -u_new[1,:]; u_new[-1,:] = -u_new[-2,:]; u_new[:,0] = -u_new[:,1]; u_new[:,-1] = -u_new[:,-2]
        v_new[0,:] = -v_new[1,:]; v_new[-1,:] = -v_new[-2,:]; v_new[:,0] = -v_new[:,1]; v_new[:,-1] = -v_new[:,-2]
        
        h[:] = h_new[:]
        u[:] = u_new[:]
        v[:] = v_new[:]
        
    plt.figure(figsize=(8, 6))
    plt.contourf(X, Y, h, 50, cmap='ocean')
    plt.colorbar(label='Water height')
    plt.title('2D Shallow Water Equations (Ripple effect)')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.tight_layout()
    plt.savefig('Fluidos_Shallow_Water.png')
    plt.show()

if __name__ == '__main__':
    shallow_water_solver()
